# Updated colocation Mar 2026

- Replaces 0.5_socatv2024_colocation.ipynb
- Better to colocate SOCAT by comparison to the final coreINDEX since some profiles are dropped from open ocean masking


In [1]:
# os tools
import os
from tqdm import tqdm

# data tools
import numpy                 as np
import pandas                as pd
import xarray                as xr
from   datetime              import date, datetime, timedelta                 # for saving figures with today's date
import datetime
import scipy
from   scipy.stats           import kruskal              # for boxenplot stats
from   scipy.stats           import mannwhitneyu         # for split violin plot stats
import gsw


# for all plots
import matplotlib
import matplotlib.pyplot     as plt                      # needed to make map setup
import matplotlib.colors     as colors
from   matplotlib.ticker     import EngFormatter         # for degree symbol in axis
from   cmocean               import cm as cmo
import seaborn               as     sns

# for map
import shapefile
import cartopy                                           # to make map
import matplotlib.path       as     mpath                # to draw circle for map
import cartopy.crs           as     ccrs                 # for map projection
import cartopy.feature       as     cfeature             # to add land features to map

import pyproj  
import geopandas             as     gpd                  # for adding shapefiles of frontal zones 
from   osgeo                 import gdal


# Custom modules
import mod_cremas as crx 
import mod_ocean as myocn

from importlib import reload
import mod_plotting as myplt

import mod_loading as loader


# === CUSTOM FUNCTIONS
import mod_preprocessing as mod_prep
import mod_ocean

plt.rcParams.update(myplt.my_params(size=12))

In [2]:
import mod_argo

In [3]:
import shapefile
so_fronts = shapefile.Reader('./shapefiles/fronts/so_fronts.shp') 
stf_mod   = shapefile.Reader('./shapefiles/fronts/stf_mod/stf_mod.shp')

stf  = stf_mod.shape(0).points
saf  = so_fronts.shape(1).points
pf   = so_fronts.shape(2).points
sacc = so_fronts.shape(3).points
sie  = so_fronts.shape(4).points

max_latitude:          float = -30
add_gridlines:         bool  = True
color_land:            bool  = False
land_edgecolor:        str   = 'grey'
land_facecolor:        str   = 'grey'
fontsize:              float = 10
map_facecolor:         str   = 'white'
coast_linewidth:       float = 0.3
gridlines_linewidth:   float = 0.5
girdlines_color:       str   = 'grey'
gridlines_alpha:       float = 0.5
longitude_label_color: str   = 'grey'
latitude_label_color:  str   = 'grey'

# Import data (updated may 2026)

- alternate socat processing


In [5]:
# this is the float data that will be clustered in 2.1
[coreDS, coreINDEX] = loader.import_core_data(type='processed') # has adt, sla, mld, pco2
[coreDS_test, coreINDEX_test] = loader.import_core_data(type='2024') # for testing with socat 2024 data, has adt, sla, mld, pco2
# [coreDS, _] = loader.import_core_data(type='L3_extended') # now includes core 2024 data for potential testing


In [6]:
coreDS = coreDS.reset_coords(['yearday', 'latitude', 'longitude', 'datetime'])
coreDS_test = coreDS_test.reset_coords(['yearday', 'latitude', 'longitude', 'datetime', 'year', 'month', 'day'])
combinedINDEX = xr.merge([coreINDEX, coreINDEX_test])

In [4]:
socatDS = loader.import_socat_L2(version='v2025') # includes socat 2024 test data

In [ ]:
# socat_trainval = pd.read_csv(folder + 'socat_trainval_processed_co2_mld_adt_yr2014-2024_acc20260326.csv', index_col=0)

# Co-location with core argo data used for clustering

In [7]:
# # troubleshooting new version
# platDF = temp
# argoINDEX = combinedINDEX
# yd_window = 7
# ref_time = '2014-01-01'
# Lx=2.5


# colocation = pd.DataFrame(index=platDF.index, columns=
#                         ['nearest_profid', 'prof_datetime', 'prof_lat', 'prof_lon',
#                             'yd_sep', 'km_sep'])
# for idx in tqdm(platDF.index):
#     lower = platDF.loc[idx, 'yearday'] - yd_window
#     upper = platDF.loc[idx, 'yearday'] + yd_window
#     argoset = (argoINDEX.where(argoINDEX.yearday > lower)
#                 .where(argoINDEX.yearday < upper)
#                 )

#     argoDF = (argoset.reset_coords().to_dataframe() # .mean(dim='pressure')
#                 .dropna(subset = ['CT', 'SA'])
#                 )

#     if len(argoDF) > 0:
#         # Recalculate distance and yearday separation for each socat obs
#         argoDF['yd_sep'] = argoDF.yearday.map(lambda x: abs(x - platDF.loc[idx, 'yearday']))
#         argoDF['km_sep'] = argoDF.apply(lambda x: gsw.distance([platDF.loc[idx,'longitude'], x.longitude], 
#                                                             [platDF.loc[idx,'latitude'], x.latitude])/1000, axis=1) # km

#         argoDF['weighted_dist'] = argoDF.apply(lambda x: mod_argo.weighted_distance(
#                                         [platDF.loc[idx,'longitude'], x.longitude], 
#                                         [platDF.loc[idx,'latitude'], x.latitude], 
#                                         Lx=Lx, Ly=1), axis=1)
#         # imin = argoDF.idxmin(skipna=True).weighted_dist
#         imin = argoDF.sort_values(by='weighted_dist').index.values[0]
        
#         # Store metrics of nearest profile ID
#         colocation.loc[idx, 'nearest_profid'] = imin
#         colocation.loc[idx, 'prof_datetime'] = myocn.ytd2datetime(argoDF.loc[imin].yearday, ref_time=ref_time)
#         colocation.loc[idx, 'prof_lat'] = argoDF.loc[imin].latitude
#         colocation.loc[idx, 'prof_lon'] = argoDF.loc[imin].longitude
#         colocation.loc[idx, 'yd_sep'] = argoDF.loc[imin].yd_sep
#         colocation.loc[idx, 'km_sep'] = argoDF.loc[imin].km_sep


In [8]:
# savepath = '../working-vars/socat/colocate-coreArgo/' #(used to be in crusoe-repo)
savepath = '/Volumes/crusoe-repo/data/socat/colocate-coreArgo/'
save = True
datetag = datetime.datetime.now().strftime("%Y%m%d")

# ====
socat_df = socatDS.copy().to_dataframe()
colocate_7d_byYear = {k:None for k in [i for i in range(2014, 2025)]}

yrlist = [2014, 2015,
          2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
for yr in yrlist:
    print('===> Processing year:', yr)
    temp = socat_df[socat_df['datetime'].astype('datetime64[ns]').dt.year == yr].reset_index()  # be sure to reset index here, at end
    colocate_7d_byYear[yr] = mod_argo.find_nearest_float(temp, combinedINDEX, yd_window=7, ref_time='2014-01-01')

    if save:
        colocate_7d_byYear[yr].to_csv(savepath + 'colocate_v2025_validArgo_7d_yr' + str(yr) + '_acc' + datetag + '.csv')
        print('Saved to path:', savepath + 'colocate_v2025_validArgo_7d_yr' + str(yr) + '_acc' + datetag + '.csv')

===> Processing year: 2014


100%|██████████| 461/461 [01:08<00:00,  6.75it/s]


Saved to path: /Volumes/crusoe-repo/data/socat/colocate-coreArgo/colocate_v2025_validArgo_7d_yr2014_acc20260518.csv
===> Processing year: 2015


100%|██████████| 706/706 [01:47<00:00,  6.55it/s]


Saved to path: /Volumes/crusoe-repo/data/socat/colocate-coreArgo/colocate_v2025_validArgo_7d_yr2015_acc20260518.csv
===> Processing year: 2016


...

Saved to path: /Volumes/crusoe-repo/data/socat/colocate-coreArgo/colocate_v2025_validArgo_7d_yr2021_acc20260518.csv
===> Processing year: 2022


100%|██████████| 403/403 [00:59<00:00,  6.72it/s]


Saved to path: /Volumes/crusoe-repo/data/socat/colocate-coreArgo/colocate_v2025_validArgo_7d_yr2022_acc20260518.csv
===> Processing year: 2023


100%|██████████| 771/771 [01:47<00:00,  7.16it/s]

Saved to path: /Volumes/crusoe-repo/data/socat/colocate-coreArgo/colocate_v2025_validArgo_7d_yr2023_acc20260518.csv


In [9]:
for yr in [2024]:
    print('===> Processing year:', yr)
    temp = socat_df[socat_df['datetime'].astype('datetime64[ns]').dt.year == yr].reset_index()  # be sure to reset index here, at end
    colocate_7d_byYear[yr] = mod_argo.find_nearest_float(temp, combinedINDEX, yd_window=7, ref_time='2014-01-01')

    if save:
        colocate_7d_byYear[yr].to_csv(savepath + 'colocate_v2025_validArgo_7d_yr' + str(yr) + '_acc' + datetag + '.csv')
        print('Saved to path:', savepath + 'colocate_v2025_validArgo_7d_yr' + str(yr) + '_acc' + datetag + '.csv')

        # can be accessed using loader.import_socat_colocation() in future scripts

===> Processing year: 2024


100%|██████████| 446/446 [01:03<00:00,  7.01it/s]

Saved to path: /Volumes/crusoe-repo/data/socat/colocate-coreArgo/colocate_v2025_validArgo_7d_yr2024_acc20260518.csv


In [ ]:
# Plot paneled, SOCAT obs by year
import mod_southpolarplot as sopo
reload(myocn)
temp = myocn.expand_datetime(socat_df, type='dataframe')

fig, axs = plt.subplots(2,5, figsize=(17, 10), layout='tight', subplot_kw={'projection': ccrs.SouthPolarStereo()})

for ax in axs.flatten():
  sopo.format_southpolar(ax,
                           add_gridlines = True,
                              color_land            = True,
                              land_facecolor        = land_facecolor,
                              land_edgecolor        = land_edgecolor,
                              fontsize              = 14,
                              max_latitude         = -35,
                              map_facecolor         = 'white',
                              coast_linewidth       = coast_linewidth)
  # sopo.add_frontlines(ax)

for ind, ax in enumerate(axs.flatten()):
    yr = ind + 2014
    data = temp[temp.year == yr]
    # data = temp.where(temp.year == yr)
    figtitle = str(yr)
    # print(len(data))
    ax.scatter(data.longitude, data.latitude, c='royalblue', s=5, alpha=0.85, transform=ccrs.PlateCarree(), label='socat', zorder=3)
    ax.set_title(figtitle)

# ax.legend()

# P1 processing: Process SOCAT fCO2 to pCO2

- Copied from 1.0_RUN_Preprocessing

In [15]:
# Ner version Mar 2026 for SOCATv2025

sepstat_7d = loader.import_socat_colocation(buffer_time='7d')
socat_df = sepstat_7d[sepstat_7d['latitude']<-35]


# Moved to loader
# import os
# filepath = '../working-vars/socat/colocate-coreArgo/'
# sepdict_7d = {key:None for key in [str(x) for x in range(2014,2025)]}

# for x in os.listdir(filepath):
#     if x.startswith('colocate_v2025_validArgo_7d') & x.endswith('20260327.csv'):
#         # sepdict_7d[x[14:18]] = pd.read_csv(filepath+x, index_col=0)
#         sepdict_7d[x[30:34]] = pd.read_csv(filepath+x, index_col=0)
#         # print('Imported data for _7d window: ' + x)

# sepstat_7d = pd.concat(sepdict_7d.values()).reset_index().drop(['level_0', 'index'], axis=1)

Imported data for _7d window: colocate_v2025_validArgo_7d_yr2014_acc20260518.csv
Imported data for _7d window: colocate_v2025_validArgo_7d_yr2015_acc20260518.csv
Imported data for _7d window: colocate_v2025_validArgo_7d_yr2016_acc20260518.csv
Imported data for _7d window: colocate_v2025_validArgo_7d_yr2017_acc20260518.csv
Imported data for _7d window: colocate_v2025_validArgo_7d_yr2018_acc20260518.csv
Imported data for _7d window: colocate_v2025_validArgo_7d_yr2019_acc20260518.csv
Imported data for _7d window: colocate_v2025_validArgo_7d_yr2020_acc20260518.csv
Imported data for _7d window: colocate_v2025_validArgo_7d_yr2021_acc20260518.csv
Imported data for _7d window: colocate_v2025_validArgo_7d_yr2022_acc20260518.csv
Imported data for _7d window: colocate_v2025_validArgo_7d_yr2023_acc20260518.csv
Imported data for _7d window: colocate_v2025_validArgo_7d_yr2024_acc20260518.csv


In [17]:
# If starting new, import socat colocation data
# socat_colocated = loader.import_socat_colocation() # 7d 

socat_df = sepstat_7d[sepstat_7d['latitude']<-35]


In [18]:
# ==== Convert SOCAT fugacity to partial pressure pCO2 (~1 min)
# Updated Feb 06 2026 # Jan 21 2026

shipDF = mod_prep.convert_socat_fco2(socat_df)
shipDF['mld'] = shipDF.nearest_profid.apply(lambda x: combinedINDEX.sel(profid=x).mld.values) #combined is core + core_test


In [19]:
combinedINDEX

<xarray.Dataset>
Dimensions:         (profid: 354858)
Coordinates:
  * profid          (profid) object '1900410_id260' ... '7902229_id004'
Data variables: (12/33)
    pressure        (profid) float64 6.4 7.0 6.0 6.6 5.9 ... 11.0 0.4 0.3 0.4
    wmoid           (profid) float64 1.9e+06 1.9e+06 ... 7.902e+06 7.902e+06
    latitude        (profid) float64 -40.36 -40.11 -40.25 ... -37.17 -36.95
    longitude       (profid) float64 95.36 96.08 96.68 ... 7.324 7.596 6.532
    datetime        (profid) object '2014-01-02 02:34:12.000000000' ... 17354...
    yearday         (profid) float64 1.107 11.27 21.44 ... 4.006e+03 4.015e+03
    ...              ...
    atmos_co2_ppm   (profid) float64 393.9 393.9 393.9 393.9 ... nan nan nan nan
    sst             (profid) float64 12.72 13.14 14.36 13.68 ... nan nan nan nan
    sss             (profid) float64 35.2 35.2 35.14 35.18 ... nan nan nan nan
    vapor_pres_atm  (profid) float64 0.01422 0.01462 0.01582 ... nan nan nan
    pco2_atmos      (profid) float64 392.0 391.1 386.1 389.0 ... nan nan nan nan
    mlp             (profid) float64 nan nan nan nan ... 12.37 30.27 50.96 29.54

In [20]:
shipDF = shipDF.loc[:,~shipDF.columns.duplicated()].copy() # if linear_time got duplicated
shipDF = mod_prep.add_regression_time_vars(shipDF)
shipDF = mod_prep.add_regression_carbon_vars(shipDF, convert_pco2=True) 
shipDF['delta_pco2'] = shipDF['pco2_ocean'] - shipDF['pco2_atmos']

# 
# shipDF

/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CRUSOE/src/mod_ocean.py:35: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (df['datetime'].dt.is_leap_year.replace({True: 366, False: 365}))


In [21]:
# Add ADT and add profid to soccom_df
print('Adding satellite ADT...')
shipDF_adt = mod_prep.add_satellite_adt(shipDF, year_range = range(2014,2025))

Adding satellite ADT...


In [ ]:
# Add solar radiation daily max
shipDF_solar = mod_prep.add_solar_radiation(shipDF_adt, daily_max=True)

In [66]:
shipDF_adt

,longitude,latitude,fco2rec,sal,sst,linear_time,fco2water_equ_wet,fco2water_sst_wet,pco2water_equ_wet,pco2water_sst_wet,...,atmos_co2_ppm,sss,vapor_pres_atm,pco2_atmos,delta_pco2,mld,year,month,day,adt
0,-24.64215,-73.72590,341.6865,34.0740,-1.2305,1.483947,NaN,341.700,NaN,343.2,...,393.99634972925736,34.405349,0.005405,385.975982,-42.756075,11.667522,2014,1,2,-1.2091
1,-36.40135,-77.10035,365.8905,34.2810,-1.7040,9.499502,NaN,365.900,NaN,367.5,...,393.92330529750296,34.614034,0.005219,385.238844,-17.695680,11.667522,2014,1,10,-0.9939
2,-38.14620,-77.89910,275.3920,34.2790,-1.4480,15.491701,NaN,275.400,NaN,276.6,...,393.84300194956154,34.612038,0.005318,383.955134,-107.323573,10.548086,2014,1,16,-0.9822
3,-38.94590,-77.60800,375.1900,34.4180,-1.7700,16.295197,NaN,375.200,NaN,376.9,...,393.8420320263694,34.752420,0.005193,384.469845,-7.583650,10.548086,2014,1,17,-1.0041
4,-28.47890,-74.55740,298.7880,33.7600,-1.3940,21.498843,NaN,298.800,NaN,300.1,...,393.75378587854846,34.088126,0.005341,390.543232,-90.411353,10.657734,2014,1,22,-1.2278
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7235,-136.38265,-73.29205,250.8455,33.2585,-1.1815,3702.136748,NaN,250.845,NaN,NaN,...,417.54551391571215,33.580067,0.005426,411.097051,-159.126567,30.405754,2024,2,20,-1.1179
7236,-99.91180,-70.46775,398.7545,33.3240,-0.9140,3705.730799,NaN,398.755,NaN,NaN,...,417.5354155547392,33.643676,0.005534,400.987695,-0.451374,49.797021,2024,2,23,-1.0880
7237,-65.76490,-60.64345,425.1250,33.7330,2.6230,3712.493576,NaN,425.125,NaN,NaN,...,417.48524665126774,34.055110,0.007141,399.609350,27.326695,42.749931,2024,3,1,-0.8436
7238,-55.87635,-44.48470,338.7715,34.0840,15.0715,3717.476505,NaN,338.770,NaN,NaN,...,417.61437556576453,34.406736,0.016576,413.803880,-73.805735,21.728755,2024,3,6,-0.0622


In [ ]:
# temp_2024 = pd.read_csv('../working-vars/regression/P1-processed/socatv2025_test_processed_co2_mld_adt_yr2024_acc20260327.csv', index_col=0)
# temp_all = pd.read_csv('../working-vars/regression/P1-processed/socatv2025_trainval_processed_co2_mld_adt_yr2014-2023_acc20260327.csv', index_col=0)

In [23]:
shipDF_wind = mod_prep.add_wind_speed(shipDF_solar, year_range = range(2014,2025))

# shipDF_wind.to_csv('../working-vars/regression/P1-processed/socatv2025_temporary_windspeed.csv')
# shipDF_wind

100%|██████████| 11/11 [03:01<00:00, 16.53s/it]


In [24]:
shipDF_ice = mod_prep.add_sea_ice(shipDF_wind, year_range = range(2014,2025))

# shipDF_ice.to_csv('../working-vars/regression/P1-processed/socatv2025_temporary_icefraction.csv')
# shipDF_ice

100%|██████████| 11/11 [00:06<00:00,  1.58it/s]


In [25]:
shipDF_ice.columns

Index(['longitude', 'latitude', 'fco2rec', 'sal', 'sst', 'linear_time',
       'fco2water_equ_wet', 'fco2water_sst_wet', 'pco2water_equ_wet',
       'pco2water_sst_wet', 'sample_depth', 'xco2water_equ_dry',
       'xco2water_sst_dry', 'datetime', 'expoID', 'bathymetry',
       'nearest_profid', 'prof_datetime', 'prof_lat', 'prof_lon', 'yd_sep',
       'km_sep', 'SA', 'CT', 'lon_round', 'lat_round', 'cruiseID',
       'atmos_pres_Pa', 'atmos_pres_atm', 'pco2_ocean', 'mld', 'ydcos',
       'ydsin', 'decimalyr', 'sin_lat', 'atmos_co2_ppm', 'sss',
       'vapor_pres_atm', 'pco2_atmos', 'delta_pco2', 'year', 'month', 'day',
       'adt', 'max_solar_rad', 'wind_components', 'wind_speed', 'sea_ice'],
      dtype='object')

# SAVE P1-processed: ship data

p1-processed has carbon variable, mld, adt

In [26]:
# shipDF_ice = pd.read_csv('../working-vars/regression/P1-processed/socatv2025_temporary_icefraction.csv')
shipDF_ice['log_mld'] = np.log(shipDF_ice['mld'])

In [28]:
use_cols = [ 'cruiseid', 'datetime',  # 'sid', 'cluster', 
    'longitude', 'latitude', 'linear_time', 'sample_depth',
    'nearest_profid', 'prof_datetime', 'prof_lat', 'prof_lon', 'yd_sep', 'km_sep', 
    'ydcos', 'ydsin',
    'CT', 'SA', 'sst', 'sss', 'mld', 'log_mld',
    'adt', 'wind_speed', 'sea_ice', 'max_solar_rad',
    'atmos_pres_Pa', 'atmos_pres_atm',  'vapor_pres_atm',
    'fco2rec', 
    'pco2_atmos', 'pco2_ocean', 'delta_pco2']

shipDF_processed = (shipDF_ice.rename(columns={'cruiseID':'cruiseid'})
                                .reset_index()[use_cols].copy())

# shipDF_processed = (shipDF_ice.reset_index()[use_cols].copy())
shipDF_processed['datetime'] = pd.to_datetime(shipDF_processed['datetime'])


print(datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
shipDF_processed


2026-05-18 11:04:52


,cruiseid,datetime,longitude,latitude,linear_time,sample_depth,nearest_profid,prof_datetime,prof_lat,prof_lon,...,wind_speed,sea_ice,max_solar_rad,atmos_pres_Pa,atmos_pres_atm,vapor_pres_atm,fco2rec,pco2_atmos,pco2_ocean,delta_pco2
0,06AQ20131221,2014-01-02 10:52:07,-24.22600,-73.72310,1.452859,NaN,7900378_id041,2014-01-09,-71.616580,-22.823985,...,4.695954,0.27,540.186358,99810.0,0.985048,0.005398,343.7170,385.978703,345.259887,-40.718816
1,06AQ20131221,2014-01-10 07:14:48,-36.40195,-77.10100,9.301944,NaN,7900378_id041,2014-01-09,-71.616580,-22.823985,...,4.160672,0.82,463.987967,99800.0,0.984949,0.005221,365.8940,385.938029,367.546589,-18.391439
2,06AQ20131221,2014-01-16 04:05:29,-37.99060,-77.89930,15.170475,NaN,7900378_id042,2014-01-19,-71.682778,-23.213771,...,1.644289,0.73,431.880726,99250.0,0.979521,0.005311,312.3075,383.686105,313.713605,-69.972500
3,06AQ20131221,2014-01-17 03:51:36,-38.93040,-77.61060,16.160833,NaN,7900378_id042,2014-01-19,-71.682778,-23.213771,...,4.427912,0.83,433.520150,99440.0,0.981396,0.005197,372.7730,384.468482,374.458061,-10.010420
4,06AQ20131221,2014-01-22 09:45:30,-28.47765,-74.57455,21.406597,NaN,7900378_id043,2014-01-29,-71.743994,-23.090656,...,5.135520,0.39,468.764600,101040.0,0.997187,0.005376,299.8790,390.529607,301.226161,-89.303446
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7385,WFCH20240206,2024-02-20 03:01:15,-136.56415,-73.33460,3702.125868,NaN,5906556_id041,2024-02-20,-70.554700,-135.543900,...,6.558754,0.0,330.551286,100310.0,0.989983,0.005417,248.8055,411.101154,249.921699,-161.179455
7386,WFCH20240206,2024-02-23 17:32:21,-99.91180,-70.46775,3705.730799,NaN,3902325_id095,2024-02-24,-67.503000,-103.499490,...,12.358945,0.0,359.673126,97870.0,0.965902,0.005534,398.7545,400.987695,400.536321,-0.451374
7387,WFCH20240206,2024-03-01 09:10:49,-65.53440,-61.33220,3712.382512,NaN,4902639_id004,2024-03-05,-60.834412,-64.436043,...,9.074726,0.0,477.667973,97720.0,0.964421,0.007012,425.7730,399.708068,427.593045,27.884977
7388,WFCH20240303,2024-03-06 10:25:30,-56.05020,-44.69030,3717.434375,NaN,3901565_id124,2024-03-12,-45.024100,-57.620200,...,1.955932,0.0,708.085367,102080.0,1.007451,0.016538,340.5840,413.813964,341.817764,-71.996199


In [29]:
trainval = shipDF_processed[shipDF_processed.datetime.dt.year < 2024].copy()
test = shipDF_processed[shipDF_processed.datetime.dt.year == 2024].copy()

In [30]:
trainval

,cruiseid,datetime,longitude,latitude,linear_time,sample_depth,nearest_profid,prof_datetime,prof_lat,prof_lon,...,wind_speed,sea_ice,max_solar_rad,atmos_pres_Pa,atmos_pres_atm,vapor_pres_atm,fco2rec,pco2_atmos,pco2_ocean,delta_pco2
0,06AQ20131221,2014-01-02 10:52:07,-24.22600,-73.72310,1.452859,NaN,7900378_id041,2014-01-09,-71.616580,-22.823985,...,4.695954,0.27,540.186358,99810.0,0.985048,0.005398,343.7170,385.978703,345.259887,-40.718816
1,06AQ20131221,2014-01-10 07:14:48,-36.40195,-77.10100,9.301944,NaN,7900378_id041,2014-01-09,-71.616580,-22.823985,...,4.160672,0.82,463.987967,99800.0,0.984949,0.005221,365.8940,385.938029,367.546589,-18.391439
2,06AQ20131221,2014-01-16 04:05:29,-37.99060,-77.89930,15.170475,NaN,7900378_id042,2014-01-19,-71.682778,-23.213771,...,1.644289,0.73,431.880726,99250.0,0.979521,0.005311,312.3075,383.686105,313.713605,-69.972500
3,06AQ20131221,2014-01-17 03:51:36,-38.93040,-77.61060,16.160833,NaN,7900378_id042,2014-01-19,-71.682778,-23.213771,...,4.427912,0.83,433.520150,99440.0,0.981396,0.005197,372.7730,384.468482,374.458061,-10.010420
4,06AQ20131221,2014-01-22 09:45:30,-28.47765,-74.57455,21.406597,NaN,7900378_id043,2014-01-29,-71.743994,-23.090656,...,5.135520,0.39,468.764600,101040.0,0.997187,0.005376,299.8790,390.529607,301.226161,-89.303446
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6939,WFCH20231213,2023-12-15 11:19:27,-68.00490,-62.26465,3635.471840,NaN,5906560_id034,2023-12-19,-62.277300,-68.640400,...,2.608996,0.0,724.721122,98320.0,0.970343,0.006200,405.6200,402.685513,407.394392,4.708879
6940,WFCH20231213,2023-12-25 13:18:15,-65.75300,-62.57140,3645.554340,NaN,5906560_id035,2023-12-30,-62.152700,-68.108100,...,12.974904,0.0,724.148893,97010.0,0.957414,0.006315,404.2690,397.216238,406.031424,8.815185
6941,WFCH20231213,2023-12-26 08:33:22,-66.03820,-57.57350,3646.356505,NaN,5906560_id035,2023-12-30,-62.152700,-68.108100,...,17.566330,0.0,790.170188,98600.0,0.973106,0.008904,383.8370,402.622757,385.405059,-17.217698
6942,WFCH20231227,2023-12-29 16:48:02,-65.03500,-56.27490,3649.700023,NaN,6903769_id146,2024-01-04,-58.725878,-69.085745,...,8.329510,0.0,809.100295,98620.0,0.973304,0.008873,386.8210,402.618824,388.402316,-14.216508


In [31]:
test

,cruiseid,datetime,longitude,latitude,linear_time,sample_depth,nearest_profid,prof_datetime,prof_lat,prof_lon,...,wind_speed,sea_ice,max_solar_rad,atmos_pres_Pa,atmos_pres_atm,vapor_pres_atm,fco2rec,pco2_atmos,pco2_ocean,delta_pco2
6944,096U20240105,2024-01-05 11:43:03,145.88070,-44.65145,3656.488229,NaN,6903215_id255,2024-01-10,-43.790252,143.833693,...,4.688736,0.0,927.096477,102460.0,1.011202,0.018563,324.1865,414.535684,325.334115,-89.201569
6945,096U20240105,2024-01-06 11:30:40,142.68145,-46.66580,3657.479630,NaN,5905501_id132,2024-01-04,-45.779513,143.122653,...,3.548280,0.0,902.703601,101470.0,1.001431,0.013753,352.3285,412.409022,353.651797,-58.757225
6946,096U20240105,2024-01-07 10:43:55,145.22545,-49.66225,3658.447164,NaN,5906209_id143,2024-01-05,-50.733000,147.216000,...,10.698237,0.0,876.170935,101160.0,0.998372,0.012057,373.5775,411.779414,375.017073,-36.762341
6947,096U20240105,2024-01-08 10:10:57,148.13610,-54.10040,3659.424271,NaN,2903453_id028,2024-01-03,-54.371100,146.883700,...,6.232736,0.0,825.203064,100740.0,0.994226,0.008911,396.1540,411.333397,397.772141,-13.561256
6948,096U20240105,2024-01-09 11:04:43,149.46030,-59.15170,3660.461609,NaN,7900637_id200,2024-01-16,-59.663702,149.284939,...,17.375364,0.0,757.824824,100070.0,0.987614,0.006141,362.0390,409.742254,363.625596,-46.116657
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7385,WFCH20240206,2024-02-20 03:01:15,-136.56415,-73.33460,3702.125868,NaN,5906556_id041,2024-02-20,-70.554700,-135.543900,...,6.558754,0.0,330.551286,100310.0,0.989983,0.005417,248.8055,411.101154,249.921699,-161.179455
7386,WFCH20240206,2024-02-23 17:32:21,-99.91180,-70.46775,3705.730799,NaN,3902325_id095,2024-02-24,-67.503000,-103.499490,...,12.358945,0.0,359.673126,97870.0,0.965902,0.005534,398.7545,400.987695,400.536321,-0.451374
7387,WFCH20240206,2024-03-01 09:10:49,-65.53440,-61.33220,3712.382512,NaN,4902639_id004,2024-03-05,-60.834412,-64.436043,...,9.074726,0.0,477.667973,97720.0,0.964421,0.007012,425.7730,399.708068,427.593045,27.884977
7388,WFCH20240303,2024-03-06 10:25:30,-56.05020,-44.69030,3717.434375,NaN,3901565_id124,2024-03-12,-45.024100,-57.620200,...,1.955932,0.0,708.085367,102080.0,1.007451,0.016538,340.5840,413.813964,341.817764,-71.996199


In [34]:
# === Optional save: 
# newest version does preprocessing before clustering step 
# all variables at "P1" stage should have mld, adt, atmospheric pco2, deltapco2 all calculated

from datetime import datetime
save = True
datetag = datetime.now().strftime('%Y%m%d') 

trainval = shipDF_processed[shipDF_processed.datetime.dt.year < 2024].copy()
test = shipDF_processed[shipDF_processed.datetime.dt.year == 2024].copy()

if save: # updated may 2026 with 23.5h resample
    filepath = '../working-vars/regression/P1-processed/'

    # Split up train/test?
    filename = 'socatv2025_trainval_processed_co2_yr2014-2023_acc' + datetag + '.csv'
    trainval.to_csv(filepath + filename)

    filename = 'socatv2025_test_processed_co2_yr2024_acc' + datetag + '.csv'
    test.to_csv(filepath + filename)

    print('Saved socat training/validation data to ' + filepath + filename)

print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))


Saved socat training/validation data to ../working-vars/regression/P1-processed/socatv2025_test_processed_co2_yr2024_acc20260518.csv
2026-05-18 11:05:29
